# Limpieza y Normalización de `resources.csv`

Decisiones aplicadas:
1. **Conservación de prefijos `res_` y `org_` y nombres `snake_case` (`resource_id`, `org_id`, `service`, `region`, `created_at`, `state`)**: garantiza el 100 % de integridad referencial en los *joins* con `customers_orgs.csv` y `usage_events_stream`.
2. **Conversión de `created_at` a fecha (`datetime`)**.
3. **Desanidado de `tags_json`**: apertura en las columnas `env`, `pii`, `costcenter`, `team` y `backup` con imputación explícita de valores faltantes (`Sin Especificar`, `Sin Asignar`, `False`).

In [1]:
import json
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
df = pd.read_csv(REPO_ROOT / "data" / "raw" / "resources.csv")

# Normalización de nombres de columnas en snake_case (conservando prefijos res_ y org_ en los valores)
df.columns = ["resource_id", "org_id", "service", "region", "created_at", "state", "tags_json"]
df["resource_id"] = df["resource_id"].str.strip()
df["org_id"] = df["org_id"].str.strip()
df["service"] = df["service"].str.strip().str.lower()
df["region"] = df["region"].str.strip().str.lower()
df["state"] = df["state"].str.strip().str.lower()

conteo_nan = df["tags_json"].isna().sum()
print(f"Total de filas: {len(df)}")
print(f"Cantidad de valores NaN en tags_json: {conteo_nan}")

df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
print(f"resource_id duplicados: {df['resource_id'].duplicated().sum()}")
print(f"org_id únicos: {df['org_id'].nunique()} (duplicados esperados por relación 1:N = {df['org_id'].duplicated().sum()})")
print(f"Servicios ({df['service'].nunique()}): {df['service'].unique().tolist()}")
print(f"Regiones ({df['region'].nunique()}): {df['region'].unique().tolist()}")
print(f"Estados ({df['state'].nunique()}): {df['state'].unique().tolist()}")

# Apertura de tags_json
df["tags_json"] = df["tags_json"].fillna("[]").apply(json.loads)

def parsear_tags_a_dict(lista_tags):
    diccionario = {}
    for tag in lista_tags:
        if ":" in tag:
            clave, valor = tag.split(":", 1)
            diccionario[clave.strip()] = valor.strip()
    return diccionario

df_tags = df["tags_json"].apply(parsear_tags_a_dict).apply(pd.Series)
df = pd.concat([df.drop(columns=["tags_json"]), df_tags], axis=1)

df["env"] = df["env"].fillna("Sin Especificar")
df["pii"] = df["pii"].map({"true": True}).fillna(False).astype(bool)
df["costcenter"] = df["costcenter"].fillna("Sin Asignar")
df["team"] = df["team"].fillna("Sin Asignar")
df["backup"] = df["backup"].map({"daily": True}).fillna(False).astype(bool)

# Ordenar columnas de forma consistente
cols_orden = ["resource_id", "org_id", "service", "region", "created_at", "state", "env", "pii", "costcenter", "team", "backup"]
df = df[cols_orden]

# Verificación de integridad referencial contra customers_orgs.csv
orgs_df = pd.read_csv(REPO_ROOT / "data" / "raw" / "customers_orgs.csv")
orgs_ok = df["org_id"].isin(orgs_df["org_id"]).sum()
print("\n--- VERIFICACIÓN DE INTEGRIDAD REFERENCIAL ---")
print(f"Org IDs de resources presentes en customers_orgs.csv: {df['org_id'].nunique()} / {orgs_df['org_id'].nunique()} ({orgs_ok / len(df) * 100:.1f}%)")

out_path = REPO_ROOT / "data" / "clean" / "resources_limpio.csv"
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Archivo guardado en: {out_path.relative_to(REPO_ROOT).as_posix()} ({len(df)} filas, {df.isna().sum().sum()} nulos)")

Total de filas: 400
Cantidad de valores NaN en tags_json: 83
resource_id duplicados: 0
org_id únicos: 80 (duplicados esperados por relación 1:N = 320)
Servicios (6): ['compute', 'database', 'storage', 'networking', 'analytics', 'genai']
Regiones (7): ['sa-east', 'ap-south', 'eu-central', 'us-west', 'ap-northeast', 'eu-west', 'us-east']
Estados (3): ['running', 'stopped', 'terminated']

--- VERIFICACIÓN DE INTEGRIDAD REFERENCIAL ---
Org IDs de resources presentes en customers_orgs.csv: 80 / 80 (100.0%)
Archivo guardado en: data/clean/resources_limpio.csv (400 filas, 0 nulos)
